# ShowPDF4D — Interactive PDF Analysis (Simulated Ta)

Interactive PDF analysis of a simulated 4D-STEM dataset of amorphous tantalum (Ta).
20×20 scan, 307×307 detector.

In [5]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


In [6]:
import quantem as em
from quantem.core.io.serialize import load
from quantem.diffraction import PairDistributionFunction
from quantem.widget import ShowPDF4D
from quantem.widget.detector import virtual_images
import numpy as np
from pathlib import Path

## Load simulated Ta 4D-STEM data

In [7]:
file_path = Path("../../../quantem-tutorials/data/Ta_sim_binned.zip")
ds = load(file_path)
print(f"4D shape: {ds.array.shape}")
print(f"Sampling: {ds.sampling}")
print(f"Units: {ds.units}")

4D shape: (20, 20, 307, 307)
Sampling: [1. 1. 1. 1.]
Units: ['pixels', 'pixels', 'pixels', 'pixels']


## Polar transform with automatic origin finding

In [8]:
origins = ds.auto_origin_id()
polar = ds.polar_transform(origin_array=origins)
polar.sampling[3] = 0.01488
print(f"Polar shape: {polar.shape}")
print(f"q range: 0 to {polar.shape[3] * polar.sampling[3]:.4f} Å⁻¹")

Finding origin for each scan position: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:04<00:00, 97.63it/s]


Polar shape: (20, 20, 180, 151)
q range: 0 to 2.2469 Å⁻¹


## Build PDF and launch widget

In [9]:
pdf = PairDistributionFunction.from_data(polar)
print(f"q range: {pdf.qq[0]:.4f} to {pdf.qq[-1]:.4f} Å⁻¹")

q range: 0.0000 to 2.2320 Å⁻¹


In [10]:
# BF virtual image for navigation panel
bf, adf, haadf = virtual_images(ds.array)
print(f"BF shape: {bf.shape}")

BF shape: (20, 20)


In [11]:
w = ShowPDF4D(
    pdf,
    nav_image=bf,
    title="Ta Simulated PDF",
    k_min_fit=0.05,
    r_max=20.0,
)
w

ShowPDF4D(scan=(20, 20), k=[0.1, 1.8])

In [12]:
w.summary()

Ta Simulated PDF
════════════════════════════════
Scan:     20 × 20
k range:  [0.00, 2.23] Å⁻¹
Fit:      k=[0.05, 1.79]
Output:   r=[0.00, 20.00], step=0.02
Plot:     Gr
Mask:     full scan (no mask)


In [13]:
# Test mask recomputation end-to-end
import base64

# 1. Record baseline G(r) with no mask
Gr_baseline = np.frombuffer(w.gr_y_bytes, dtype=np.float32).copy()
print(f"Baseline G(r): {len(Gr_baseline)} points, range [{Gr_baseline.min():.4f}, {Gr_baseline.max():.4f}]")

# 2. Set a mask programmatically — select only top-left 10x10 quadrant
mask = np.zeros((20, 20), dtype=np.uint8)
mask[:10, :10] = 1
w.mask_b64 = base64.b64encode(mask.tobytes()).decode()
w.mask_version += 1

import time; time.sleep(2)  # wait for recompute

# 3. Check that G(r) changed
Gr_masked = np.frombuffer(w.gr_y_bytes, dtype=np.float32).copy()
print(f"Masked G(r):   {len(Gr_masked)} points, range [{Gr_masked.min():.4f}, {Gr_masked.max():.4f}]")
print(f"Mask stats:    {w.mask_pixel_count} / 400 included ({w.mask_fraction*100:.1f}%)")

changed = not np.allclose(Gr_baseline, Gr_masked, atol=1e-6)
print(f"\nG(r) changed after masking: {changed}")
assert changed, "FAIL: G(r) did not change after applying mask!"
print("PASS: Mask recomputation works end-to-end")

Baseline G(r): 1000 points, range [-1.8260, 3.1675]
Masked G(r):   1000 points, range [-1.7019, 2.9481]
Mask stats:    100 / 400 included (25.0%)

G(r) changed after masking: True
PASS: Mask recomputation works end-to-end


In [ ]:
# 4. Clear mask and verify it reverts (approximately)
w.mask_b64 = ""
w.mask_version += 1
time.sleep(2)

Gr_cleared = np.frombuffer(w.gr_y_bytes, dtype=np.float32).copy()
max_diff = np.max(np.abs(Gr_baseline - Gr_cleared))
print(f"Mask stats after clear: {w.mask_pixel_count} / 400 ({w.mask_fraction*100:.1f}%)")
print(f"Max diff from baseline: {max_diff:.6e}")

# Masked G(r) should differ much more from baseline than cleared G(r)
masked_diff = np.max(np.abs(Gr_baseline - Gr_masked))
print(f"Max diff (masked vs baseline): {masked_diff:.6e}")
print(f"Max diff (cleared vs baseline): {max_diff:.6e}")
assert max_diff < masked_diff, "FAIL: cleared G(r) should be closer to baseline than masked G(r)"
print("PASS: Mask clear brings G(r) back close to baseline")